# 🌱 Crop Disease Detection & Farm Advisory System

## How to Run This Project

Step 1: Download the dataset from Kaggle:
https://www.kaggle.com/datasets/emmarex/plantdisease

Step 2: Click the download button to get the ZIP file.

Step 3: Run the next code cell.
A file upload window will appear.

Step 4: Upload the downloaded ZIP file (DO NOT extract it).

Step 5: Wait for extraction and model training to begin automatically.

Note:
Upload only the ZIP file containing plant disease images.

In [ ]:
!pip install tensorflow opencv-python pillow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

from google.colab import files
uploaded = files.upload()

import zipfile

zip_path = "plantvillage.zip"   # your uploaded file name
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

    import os
os.listdir("/content/dataset")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "/content/dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/content/dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

model.save("crop_disease_model.h5")

from google.colab import files
files.download("crop_disease_model.h5")

uploaded = files.upload()

from tensorflow.keras.preprocessing import image

img_path = list(uploaded.keys())[0]

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

class_index = np.argmax(prediction)
confidence = np.max(prediction)

print("Predicted Class:", class_index)
print("Confidence:", confidence)


TypeError: 'NoneType' object is not subscriptable